In [1]:
# %%
from __future__ import annotations

import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from joblib import Parallel, delayed
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline

import mne
mne.set_log_level("WARNING")

In [2]:
# %%
PROJECT_ROOT = Path("..").resolve()
DERIVED_ROOT  = PROJECT_ROOT / "data" / "derived"
MANIFEST_PATH = DERIVED_ROOT / "manifests" / "manifest_spontaneous_validated.csv"
FEAT_A_PATH   = DERIVED_ROOT / "features" / "features_A_bandpower_epochwise.csv"
FEAT_B_PATH   = DERIVED_ROOT / "features" / "features_B_wpli_edges_epoch_sliding.csv"
PARAMS_PATH   = PROJECT_ROOT / "results" / "models" / "best_params_rf_nested_groupkfold_Aepoch_BwpliEpochSliding.csv"
FIGURE_DIR    = PROJECT_ROOT / "results" / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME    = "B_wpli_edges_epoch_sliding"
N_CHANNELS    = 62
BANDS         = ["theta", "alpha", "beta"]
N_EDGES       = N_CHANNELS * (N_CHANNELS - 1) // 2   # 1891
OUTER_SPLITS  = 5
PCA_N_COMPONENTS = 100
SEED          = 0

N_CORES = os.cpu_count() or 8
print(f"Cores available: {N_CORES}")
print(f"Edges per band:  {N_EDGES}  |  Total B features: {len(BANDS) * N_EDGES}")

Cores available: 60
Edges per band:  1891  |  Total B features: 5673


In [3]:
# %%
# ============================================
# Section 1. Load features B and reconstruct XB / labels / groups
# (mirrors the model notebook exactly so we get identical splits)
# ============================================

A = pd.read_csv(FEAT_A_PATH)
B = pd.read_csv(FEAT_B_PATH)

A_ok = A[A.get("extract_ok", True) == True].copy()
B_ok = B[B.get("extract_ok", True) == True].copy()

key_cols = ["subject_id", "recording_number", "epoch_index_original"]

A_keep = A_ok.rename(columns={c: f"A__{c}" for c in A_ok.columns if c not in key_cols})
B_keep = B_ok.drop(columns=["drug"], errors="ignore")
B_keep = B_keep.rename(columns={c: f"B__{c}" for c in B_keep.columns if c not in key_cols})

df = A_keep.merge(B_keep, on=key_cols, how="inner")
df["drug"] = df["A__drug"]

EXCLUDE_EXACT = {
    "A__extract_ok", "B__extract_ok", "A__sfreq", "B__sfreq",
    "A__n_channels", "B__n_channels", "A__epoch_len_sec", "B__epoch_len_sec",
    "A__n_epochs_before", "B__n_epochs_before", "A__n_epochs_after", "B__n_epochs_after",
    "B__n_windows", "B__win_len_sec", "B__win_step_sec",
}
EXCLUDE_CONTAINS = ["file_path", "extract_error", "drug", "eyes",
                    "subject_id", "recording_number", "epoch_index"]

def select_feature_cols(prefix: str) -> list[str]:
    return [
        c for c in df.columns
        if c.startswith(prefix)
        and c not in EXCLUDE_EXACT
        and not any(x in c for x in EXCLUDE_CONTAINS)
        and pd.api.types.is_numeric_dtype(df[c])
    ]

B_feat_cols = select_feature_cols("B__")
XB = df[B_feat_cols].to_numpy()
y  = (df["drug"] == "ketamine").astype(int).to_numpy()
groups = df["subject_id"].astype(str).to_numpy()

# Strip the "B__" prefix to get bare column names (band_eXXXX)
bare_feat_names = [c.replace("B__", "", 1) for c in B_feat_cols]

print(f"XB shape: {XB.shape}")
print(f"Example feature names: {bare_feat_names[:3]}  …  {bare_feat_names[-3:]}")

XB shape: (276, 5674)
Example feature names: ['ptp_uv', 'theta_e0000', 'theta_e0001']  …  ['beta_e1888', 'beta_e1889', 'beta_e1890']


In [4]:
# %%
# ============================================
# Section 2. Load channel info (for topomaps)
# ============================================

manifest = pd.read_csv(MANIFEST_PATH)
sample_fp = manifest["file_path"].iloc[0]
sample_epochs = mne.io.read_epochs_eeglab(sample_fp, verbose="ERROR")
ch_info = sample_epochs.info
ch_names = sample_epochs.ch_names
print(f"Channels ({len(ch_names)}): {ch_names[:5]} … {ch_names[-3:]}")

Channels (62): ['Iz', 'O2', 'Oz', 'O1', 'PO8'] … ['Fp2', 'Fpz', 'Fp1']


In [5]:
# %%
# ============================================
# Section 3. Load saved best params for model B
# ============================================

params_df = pd.read_csv(PARAMS_PATH)
B_params = params_df[params_df["model"] == MODEL_NAME].copy()

def _coerce(v):
    if isinstance(v, float) and np.isnan(v):
        return None
    if isinstance(v, float) and v.is_integer():
        return int(v)
    return v

def _coerce_max_features(v):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return None
    if isinstance(v, str) and v.strip() in {"sqrt", "log2"}:
        return v.strip()
    try:
        f = float(v)
        return int(f) if f.is_integer() else f
    except Exception:
        return v

def build_pipeline_from_row(row: pd.Series) -> Pipeline:
    pca_n = int(row.get("pca_n_components", PCA_N_COMPONENTS))
    rf_params = {
        "n_estimators":      _coerce(row.get("rf__n_estimators")),
        "max_depth":         _coerce(row.get("rf__max_depth")),
        "min_samples_split": _coerce(row.get("rf__min_samples_split")),
        "min_samples_leaf":  _coerce(row.get("rf__min_samples_leaf")),
        "max_features":      _coerce_max_features(row.get("rf__max_features")),
    }
    rf_params = {k: v for k, v in rf_params.items() if v is not None}
    pipe = Pipeline([
        ("pca", PCA(n_components=pca_n, svd_solver="randomized", random_state=SEED)),
        ("rf",  RandomForestClassifier(random_state=SEED, class_weight="balanced_subsample",
                                       n_jobs=1, **rf_params)),
    ])
    return pipe

print(B_params[["fold", "pca_n_components", "rf__n_estimators", "rf__max_depth",
               "rf__max_features", "rf__min_samples_leaf", "rf__min_samples_split"]].to_string())

   fold  pca_n_components  rf__n_estimators  rf__max_depth rf__max_features  rf__min_samples_leaf  rf__min_samples_split
5     1             100.0             519.0           10.0             0.05                   2.0                    9.0
6     2             100.0             324.0           10.0              0.2                   4.0                    6.0
7     3             100.0             456.0           20.0             sqrt                   2.0                    5.0
8     4             100.0             295.0            NaN              0.2                   3.0                    5.0
9     5             100.0             631.0            NaN             0.05                   4.0                    3.0


In [6]:
# %%
# ============================================
# Section 4. Refit PCA+RF per outer fold (parallelized)
# Returns: PCA components + RF feature importances for each fold
# ============================================

outer_cv = GroupKFold(n_splits=OUTER_SPLITS)
splits = list(outer_cv.split(XB, y, groups))

def fit_one_fold(fold_idx: int, tr: np.ndarray) -> dict:
    row = B_params[B_params["fold"] == fold_idx].iloc[0]
    pipe = build_pipeline_from_row(row)
    pipe.fit(XB[tr], y[tr])

    pca: PCA = pipe.named_steps["pca"]
    rf: RandomForestClassifier = pipe.named_steps["rf"]

    return {
        "fold":              fold_idx,
        "components":        pca.components_.copy(),          # (n_components, n_features)
        "explained_var":     pca.explained_variance_ratio_.copy(),
        "rf_importances":    rf.feature_importances_.copy(),  # (n_components,)
    }

fold_results = Parallel(n_jobs=min(N_CORES, OUTER_SPLITS), prefer="processes")(
    delayed(fit_one_fold)(fold_idx, tr)
    for fold_idx, (tr, _) in enumerate(splits, start=1)
)
fold_results.sort(key=lambda d: d["fold"])
print(f"Fitted {len(fold_results)} folds.")
print(f"Components shape (one fold): {fold_results[0]['components'].shape}")

Fitted 5 folds.
Components shape (one fold): (100, 5674)


In [7]:
# %%
# ============================================
# Section 5. Back-project RF importances → original 5673-edge space
# Method: importance_j = sum_k( rf_importance_k * components_[k,j]^2 )
# Squared loadings avoid sign-flip cancellations across folds.
# Average across folds for stability.
# ============================================

backproj_per_fold = np.stack([
    (fr["components"] ** 2).T @ fr["rf_importances"]   # (n_features,)
    for fr in fold_results
])  # (n_folds, n_features)

importance_mean = backproj_per_fold.mean(axis=0)  # (n_features,)
importance_std  = backproj_per_fold.std(axis=0)

# Split by band
def band_importance(band: str) -> np.ndarray:
    idxs = [i for i, n in enumerate(bare_feat_names) if n.startswith(band + "_")]
    assert len(idxs) == N_EDGES, f"Expected {N_EDGES} edges for {band}, got {len(idxs)}"
    return importance_mean[idxs]

band_imp = {b: band_importance(b) for b in BANDS}
print("Importance per band (sum):", {b: float(v.sum()) for b, v in band_imp.items()})

# Edge-index → (row, col) in 62×62 matrix
triu_rows, triu_cols = np.triu_indices(N_CHANNELS, k=1)  # length 1891

Importance per band (sum): {'theta': 0.33196413150117576, 'alpha': 0.47869975191198644, 'beta': 0.1503780886633785}


In [8]:
# %%
# ============================================
# Section 6. Plot 1 — PC importance bar chart
# Shows which PCA components the RF relies on most (averaged across folds)
# ============================================

rf_imp_per_fold = np.stack([fr["rf_importances"] for fr in fold_results])  # (n_folds, n_pcs)
rf_imp_mean = rf_imp_per_fold.mean(axis=0)
rf_imp_sem  = rf_imp_per_fold.std(axis=0) / np.sqrt(len(fold_results))

ev_mean = np.stack([fr["explained_var"] for fr in fold_results]).mean(axis=0)

top_k = 20
order = np.argsort(rf_imp_mean)[::-1][:top_k]

fig, axes = plt.subplots(2, 1, figsize=(12, 8))

# Top panel: RF importance per PC
ax = axes[0]
bars = ax.bar(range(top_k), rf_imp_mean[order], color="steelblue", alpha=0.8)
ax.errorbar(range(top_k), rf_imp_mean[order], yerr=rf_imp_sem[order],
            fmt="none", color="black", capsize=3, linewidth=1)
ax.set_xticks(range(top_k))
ax.set_xticklabels([f"PC{o+1}" for o in order], rotation=45, ha="right")
ax.set_ylabel("RF feature importance (mean ± SEM across folds)")
ax.set_title(f"Top {top_k} PCA components by RF importance — model B (wPLI)")

# Bottom panel: corresponding explained variance
ax2 = axes[1]
ax2.bar(range(top_k), ev_mean[order] * 100, color="coral", alpha=0.8)
ax2.set_xticks(range(top_k))
ax2.set_xticklabels([f"PC{o+1}" for o in order], rotation=45, ha="right")
ax2.set_ylabel("Explained variance (%)")
ax2.set_title("Explained variance of the same PCs")

fig.tight_layout()
out = FIGURE_DIR / "feat_B_pc_importance.png"
fig.savefig(out, dpi=150, bbox_inches="tight")
print("Saved:", out)
plt.close(fig)

Saved: /data/storage-occ-v2/repos/ketamine-prediction/ketamine-eeg-prediction/.claude/worktrees/cool-grothendieck-d69cd2/results/figures/feat_B_pc_importance.png


In [9]:
# %%
# ============================================
# Section 7. Plot 2 — Top PC connectivity patterns
# Show the top 3 PCs (by mean RF importance) as connectivity matrices
# and corresponding node-strength topomaps
# ============================================

top3_pc_idxs = order[:3]  # indices into PC array (0-based)

# Average loadings across folds (handle sign ambiguity by aligning to fold-0)
aligned = []
ref = fold_results[0]["components"]
for fr in fold_results:
    comp = fr["components"].copy()
    for k in range(comp.shape[0]):
        if np.dot(comp[k], ref[k]) < 0:
            comp[k] *= -1
    aligned.append(comp)
mean_components = np.stack(aligned).mean(axis=0)  # (n_components, n_features)

def edge_vec_to_matrix(vec: np.ndarray) -> np.ndarray:
    mat = np.zeros((N_CHANNELS, N_CHANNELS))
    mat[triu_rows, triu_cols] = vec
    mat[triu_cols, triu_rows] = vec
    return mat

# PC loadings cover all 5673 features (3 bands × 1891)
def pc_loading_for_band(pc_idx: int, band: str) -> np.ndarray:
    idxs = [i for i, n in enumerate(bare_feat_names) if n.startswith(band + "_")]
    return mean_components[pc_idx, idxs]

fig, axes = plt.subplots(len(top3_pc_idxs), len(BANDS) + 1,
                          figsize=(5 * (len(BANDS) + 1), 4 * len(top3_pc_idxs)))

for row_i, pc_idx in enumerate(top3_pc_idxs):
    node_strength = np.zeros(N_CHANNELS)

    for col_i, band in enumerate(BANDS):
        loading = pc_loading_for_band(pc_idx, band)
        mat = edge_vec_to_matrix(loading)

        ax = axes[row_i, col_i]
        vabs = np.abs(mat).max()
        im = ax.imshow(mat, cmap="RdBu_r", vmin=-vabs, vmax=vabs, aspect="auto")
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        if row_i == 0:
            ax.set_title(band.capitalize(), fontsize=11)
        if col_i == 0:
            ax.set_ylabel(f"PC{pc_idx+1}\n(RF imp: {rf_imp_mean[pc_idx]:.4f})",
                          fontsize=10)
        ax.set_xlabel("Channel index")

        node_strength += np.abs(mat).sum(axis=1)

    # Topomap of node strength (summed across bands)
    ax_topo = axes[row_i, -1]
    im_t, _ = mne.viz.plot_topomap(
        node_strength, ch_info,
        axes=ax_topo, show=False,
        cmap="Reds", vlim=(0, node_strength.max()),
        contours=4,
    )
    plt.colorbar(im_t, ax=ax_topo, fraction=0.046, pad=0.04)
    if row_i == 0:
        ax_topo.set_title("Node strength\n(sum bands)", fontsize=11)

fig.suptitle("Top 3 PCA components — connectivity loadings & node strength",
             fontsize=13, y=1.01)
fig.tight_layout()
out = FIGURE_DIR / "feat_B_top_pc_patterns.png"
fig.savefig(out, dpi=150, bbox_inches="tight")
print("Saved:", out)
plt.close(fig)

Saved: /data/storage-occ-v2/repos/ketamine-prediction/ketamine-eeg-prediction/.claude/worktrees/cool-grothendieck-d69cd2/results/figures/feat_B_top_pc_patterns.png


In [10]:
# %%
# ============================================
# Section 8. Plot 3 — Back-projected edge importance per band
# Full 62×62 heatmap of aggregated original-space feature importance
# ============================================

fig, axes = plt.subplots(1, len(BANDS), figsize=(6 * len(BANDS), 5))

for ax, band in zip(axes, BANDS):
    imp_vec = band_imp[band]                     # (1891,)
    mat = edge_vec_to_matrix(imp_vec)            # (62, 62), symmetric

    im = ax.imshow(mat, cmap="hot", aspect="auto")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="Importance")
    ax.set_xlabel("Channel index")
    ax.set_ylabel("Channel index")

fig.tight_layout()
out = FIGURE_DIR / "feat_B_edge_importance_heatmap.png"
fig.savefig(out, dpi=150, bbox_inches="tight")
print("Saved:", out)
plt.close(fig)

Saved: /data/storage-occ-v2/repos/ketamine-prediction/ketamine-eeg-prediction/.claude/worktrees/cool-grothendieck-d69cd2/results/figures/feat_B_edge_importance_heatmap.png


In [11]:
# %%
# ============================================
# Section 9. Plot 4 — Node strength topomaps (back-projected importance)
# Sum importance across all edges incident on each node, per band
# ============================================

fig, axes = plt.subplots(1, len(BANDS), figsize=(5 * len(BANDS), 4))

for ax, band in zip(axes, BANDS):
    imp_vec  = band_imp[band]
    mat      = edge_vec_to_matrix(imp_vec)
    node_str = mat.sum(axis=1)   # (62,) — total importance weight per channel

    im, _ = mne.viz.plot_topomap(
        node_str, ch_info,
        axes=ax, show=False,
        cmap="hot",
        vlim=(0, node_str.max()),
        contours=5,
    )
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set_title(f"{band.capitalize()} band", fontsize=12)

fig.suptitle("Node strength of back-projected edge importance per band\n"
             "(higher = channel involved in more important wPLI edges)",
             fontsize=12, y=1.02)
fig.tight_layout()
out = FIGURE_DIR / "feat_B_node_strength_topomap.png"
fig.savefig(out, dpi=150, bbox_inches="tight")
print("Saved:", out)
plt.close(fig)

Saved: /data/storage-occ-v2/repos/ketamine-prediction/ketamine-eeg-prediction/.claude/worktrees/cool-grothendieck-d69cd2/results/figures/feat_B_node_strength_topomap.png


In [12]:
# %%
# ============================================
# Section 10. Summary table — top 20 most important edges (all bands)
# ============================================

records = []
for band in BANDS:
    imp_vec = band_imp[band]
    for edge_idx, (ri, ci) in enumerate(zip(triu_rows, triu_cols)):
        records.append({
            "band":    band,
            "edge":    edge_idx,
            "ch_a":    ri,
            "ch_b":    ci,
            "ch_a_name": ch_names[ri],
            "ch_b_name": ch_names[ci],
            "importance": float(imp_vec[edge_idx]),
        })

edge_df = pd.DataFrame(records)
top20 = edge_df.nlargest(20, "importance")[["band", "ch_a_name", "ch_b_name", "importance"]]
print("Top 20 most important edges (wPLI, back-projected RF importance):")
display(top20.reset_index(drop=True))

# Save full table
edge_df.to_csv(PROJECT_ROOT / "results" / "feat_B_edge_importance_table.csv", index=False)
print("Full table saved to results/feat_B_edge_importance_table.csv")

Top 20 most important edges (wPLI, back-projected RF importance):


,band,ch_a_name,ch_b_name,importance
0,alpha,Oz,PO7,0.000437
1,alpha,PO4,POz,0.000424
2,alpha,O1,PO7,0.000416
3,alpha,Oz,O1,0.000403
4,alpha,P8,FC3,0.000396
5,alpha,POz,FC6,0.000393
6,alpha,CP4,CP2,0.000389
7,alpha,O2,PO8,0.000387
8,alpha,POz,Fp2,0.000382
9,alpha,PO8,FC3,0.000381


Full table saved to results/feat_B_edge_importance_table.csv


In [13]:
# %%
print("All figures saved to:", FIGURE_DIR)
for f in sorted(FIGURE_DIR.glob("feat_B_*.png")):
    print(" ", f.name)

All figures saved to: /data/storage-occ-v2/repos/ketamine-prediction/ketamine-eeg-prediction/.claude/worktrees/cool-grothendieck-d69cd2/results/figures
  feat_B_edge_importance_heatmap.png
  feat_B_node_strength_topomap.png
  feat_B_pc_importance.png
  feat_B_top_pc_patterns.png


In [14]:
# %%
# ============================================
# Supplementary Table: top 20 back-projected wPLI edges
# ============================================
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path("..").resolve()
EDGE_TABLE_PATH = PROJECT_ROOT / "results" / "feat_B_edge_importance_table.csv"

edge_df = pd.read_csv(EDGE_TABLE_PATH)

# Top 20 edges by importance, across all bands
top20 = (edge_df
         .nlargest(20, "importance")
         [["band", "ch_a_name", "ch_b_name", "importance"]]
         .reset_index(drop=True))
top20.index = top20.index + 1   # 1-based ranking for the table

# Print as a readable preview
print("Top-20 back-projected wPLI edges (by mean importance across folds):")
print(top20.to_string(float_format="%.6f"))

# Per-band count among the top 20 (so we can describe it in prose)
print("\nBand composition of the top 20:")
print(top20["band"].value_counts())

# ---- Generate the LaTeX tabular body ----
def fmt_edge_row(rank, row):
    band = row["band"].capitalize()
    a, b = row["ch_a_name"], row["ch_b_name"]
    imp = row["importance"]
    return f"{rank} & {band} & {a}--{b} & {imp:.2e} \\\\"

print("\n" + "=" * 60)
print("LaTeX tabular body (paste into the table skeleton below)")
print("=" * 60)
for rank, row in top20.iterrows():
    print(fmt_edge_row(rank, row))

Top-20 back-projected wPLI edges (by mean importance across folds):
     band ch_a_name ch_b_name  importance
1   alpha        Oz       PO7    0.000437
2   alpha       PO4       POz    0.000424
3   alpha        O1       PO7    0.000416
4   alpha        Oz        O1    0.000403
5   alpha        P8       FC3    0.000396
6   alpha       POz       FC6    0.000393
7   alpha       CP4       CP2    0.000389
8   alpha        O2       PO8    0.000387
9   alpha       POz       Fp2    0.000382
10  alpha       PO8       FC3    0.000381
11  alpha        O1       FC4    0.000378
12  alpha        O2       FC3    0.000377
13  alpha        Iz        C1    0.000373
14  alpha        Iz        Cz    0.000372
15  alpha        P1       FT8    0.000371
16  alpha        P8        C3    0.000368
17  alpha        P8        C6    0.000367
18  alpha       POz        P6    0.000365
19  alpha       PO4        P6    0.000364
20  alpha        P3       FC6    0.000362

Band composition of the top 20:
band
alpha    20
